# Week 03 — Calibration

**Goal.** Implement ECE and reliability diagrams from scratch, then break a model's calibration on purpose and repair it.

**Deliverable.** A working `adslab.calibration` module, before/after reliability diagrams, and the sampling-rate correction derived by hand.

**Rough shape of the week.** 2h reading (Guo, Kumar) · 5h building · 2h write-up.

---
### Ground rules (they apply every week)

1. **Beat a dumb baseline or it didn't happen.** Logistic regression or the global mean.
   Log the baseline in the same table as the fancy model.
2. **Split by time, never at random.** `split.time_split` — and call
   `split.check_no_leakage` so the assertion, not your memory, enforces it.
3. **Log every run** with `registry.log_result(...)`, including the ones that lost.
   The losing runs are what make the write-up honest.
4. **Write the finding down** in this week's `README.md` while it is fresh.

### Reading

PDFs are in `papers/` next to this notebook — see `papers/README.md`.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from adslab import data, metrics, plots, split, registry, encoders, calibration

plots.use_style()
pd.set_option("display.width", 140, "display.max_columns", 60)
print("harness ready")

## Why this week is the highest-leverage one in the course

AUC cannot see calibration. A model that multiplies every prediction by 3 has *identical*
AUC and will overbid every auction by 3x. Ranking metrics are what get published;
calibration is what gets billed.

This week you fill in the stubs in `adslab/metrics.py` and `adslab/calibration.py`. The
contract is already written — `tests/test_harness.py` has an `xfail` test for each one.
Implement, run `python -m pytest -q`, and delete the `xfail` marker when it XPASSes.

In [ ]:
!cd .. && python -m pytest tests -q

## 1. Look at the predictions first

Before any metric: where do the predictions live? On a 0.2% base rate almost all of the
mass is below 0.02, which is why equal-width bins are useless here and why the default
`strategy` in `reliability_curve` is `"quantile"`.

In [ ]:
# Load a model's predictions from Week 1 or 2 (or refit quickly here).
# y_val, p_val, y_test, p_test = ...

fig, ax = plt.subplots()
plots.prediction_histogram(p_val, ax=ax)
print(plots.save(fig, 3, "prediction_distribution"))

## 2. Implement `reliability_curve`, `ece`, `max_calibration_error`

In `adslab/metrics.py`. Requirements are in the docstrings; the tests enforce them.

Then read `papers/kumar2019-verified-uncertainty-calibration.pdf` and answer in the
README: **is your ECE an underestimate, and by roughly how much?** Compute it at
`n_bins` in {10, 20, 50, 100, 200} and plot ECE against bin count. If the curve is
still rising at 200 bins, your ECE is a lower bound and you should report it as one.

In [ ]:
# after implementing:
# for nb in (10, 20, 50, 100, 200):
#     print(nb, metrics.ece(y_test, p_test, n_bins=nb))

## 3. Reliability diagram

`plots.reliability_diagram` starts working as soon as `reliability_curve` does.

Read it as a bidder: points below the diagonal mean you will overbid.

In [ ]:
# fig, ax = plt.subplots()
# plots.reliability_diagram(y_test, p_test, label="uncalibrated", ax=ax)
# print(plots.save(fig, 3, "reliability_uncalibrated"))

## 4. Platt and isotonic

Implement both in `adslab/calibration.py`. **Fit on validation, evaluate on test.**
Fitting a calibrator on training predictions corrects a distortion that does not exist
at serving time and makes test calibration worse — do it once deliberately and record
the number, because seeing it is how you remember it.

Then check what isotonic costs you in AUC. It produces ties, and ties destroy
fine-grained ranking. If AUC drops, you have found the real trade-off.

In [ ]:
# platt = calibration.PlattScaler().fit(p_val, y_val)
# iso   = calibration.IsotonicCalibrator().fit(p_val, y_val)
# for name, cal in [("platt", platt), ("isotonic", iso)]:
#     pc = cal.transform(p_test)
#     print(name, metrics.evaluate(y_test, pc))

## 5. Break it on purpose, then fix it in closed form

The stress test from the plan: **drop 50% of the positives** from training (simulating
conversion loss), retrain, and watch calibration collapse while AUC barely moves. That
divergence is the entire argument for this week.

Then fix it with a known sampling rate. Derive the correction yourself — the formula in
`SamplingRateCorrector`'s docstring is for dropped *negatives*, and the positive-dropping
case is **not** symmetric. Getting this derivation right is what makes Week 5 easy,
because privacy-driven conversion loss has exactly this shape.

In [ ]:
# 1. subsample positives in train at rate r
# 2. retrain the same model
# 3. evaluate on the UNMODIFIED test set
# 4. compare auc (barely moves) vs calibration_ratio (blows up)
# 5. apply your correction, re-evaluate

---
## Log the results

Every model you tried, including the baseline and including the failures. `notes` is the
one sentence you would say out loud about the run — future-you assembles the write-up
from these, so write it now while you still remember why the run mattered.

In [ ]:
# registry.log_result(
#     week=3,
#     model="lightgbm_hashed_2^18",
#     metrics=metrics.evaluate(y_test, p_test),
#     dataset="attribution",
#     params=dict(n_bits=18, num_leaves=63, lr=0.05),
#     notes="beats LR by 0.011 AUC; most of the gain is from cat3 x cat7 interactions",
# )

print(registry.to_markdown(week=3))

---
## Write it up

Open `README.md` in this folder and fill in the three sections. Keep it to a page.

- **What I built** — one paragraph, no code.
- **What the numbers say** — paste the table above; say which comparison is the honest one.
- **What surprised me** — the part worth reading. If nothing surprised you, you probably
  did not stress the model hard enough.

Then commit:

```bash
git add week03_* results/
git commit -m "week 03: <the finding, not the task>"
```